# Table of Contents
* [Section 1: Imports & Configuration](#imports)
* [Section 2: Extracting Words With Their Timecodes](#extract-words)
* [Section 3: Mapping Words To Phonemes](#map-words)
* [Section 4: Rhyme Unit Extraction](#extract-rhyme-units)
* [Section 5: Pairwise Rhyme Similarity](#pairwise-similarity)
* [Section 6: Words Clustering](#clustering)
* [Section 7: HTML Generation](#html-generation)
* [Section 8: Syllable Engine (boundary-free) — primary view](#syllable-engine)

---
## Section 1: Imports & Configuration <a id="imports"></a>
---

In [ ]:
import sys
sys.path.append("..")
print(sys.executable)

from python.rhyme_extraction import (
    extract_rhyme_unit, 
    extract_end_words, 
    extract_rhyme_candidates,
    extract_syllable_candidates
)

from python.similarity_engine import (
    rhyme_similarity,
    compute_similarity_pairs,
    build_similarity_matrix,
    compute_syllable_pairs,
    build_syllable_matrix
)
from python.clustering import (
    SIMILARITY_THRESHOLD,
    get_threshold,
    cluster_rhymes, 
    cluster_exact_rhymes
)
from python.debugging import debug_rhyme_analysis
from python.evaluation import evaluate_rhyme_detection
from python.html_generation import generate_rhyme_html, generate_syllable_html
from python.score_logging import log_scored_pairs

from praatio import textgrid

RHYME_MODE = "stressed"
DETECTION_MODE = "full_line"
# Options for RHYME_MODE:   "stressed" / "stressed_plus" / "entire_word"
# Options for DETECTION_MODE: "end_only" / "full_line"

INPUT_TXT_PATH = "data/input/current_input/input.txt"
OUTPUT_TEXTGRID_PATH = "data/output/current_output/input.TextGrid"

# Milestone 19a — passive pair-logging dataset
VERSE_ID = "eminem-load-the-clip"   # identifies the aligned verse for the M19 dataset
DATASET_PATH = "data/scored_pairs.jsonl"

# Milestone 13 — syllable engine clustering threshold. Re-swept on the verse
# after adopting effective-length normalisation: 0.65 keeps both cross-word wins
# (clip~gripped, load~both) while giving tight, well-separated clusters.
SYLLABLE_THRESHOLD = 0.65

---
## Section 2: Extracting Words With Their Timecodes <a id="extract-words"></a>
---

In [ ]:
tg = textgrid.openTextgrid(OUTPUT_TEXTGRID_PATH, includeEmptyIntervals=False)

word_tier = tg.getTier("words")
phone_tier = tg.getTier("phones")

for word in word_tier.entries:
    print(word)

---
## Section 3: Mapping Words To Phonemes <a id="map-words"></a>
---

In [ ]:
for word in word_tier.entries:
    word_start, word_end, word_label = word.start, word.end, word.label
    phonemes = []
    for phone in phone_tier.entries:
        if phone.start >= word_start and phone.end <= word_end:
            phonemes.append(phone.label)
    print(word_label + ":", phonemes)

# Populating the word_to_phonemes dictionary
word_to_phonemes = {}

for word in word_tier.entries:
    word_start, word_end, word_label = word.start, word.end, word.label
    
    phonemes = []
    
    for phone in phone_tier.entries:
        if phone.start >= word_start and phone.end <= word_end:
            phonemes.append(phone.label)
    
    word_label = word_label.lower()
    
    if word_label not in word_to_phonemes:
        word_to_phonemes[word_label] = phonemes

print(word_to_phonemes)

---
## Section 4: Rhyme Unit Extraction <a id="extract-rhyme-units"></a>
---

In [ ]:
rhyme_candidates = extract_rhyme_candidates(
    INPUT_TXT_PATH,
    word_to_phonemes,
    rhyme_mode=RHYME_MODE,
    detection_mode=DETECTION_MODE
)

for r in rhyme_candidates:
    print(r)

---
## Section 5: Pairwise Rhyme Similarity <a id="pairwise-similarity"></a>
---

In [ ]:
similarity_pairs = compute_similarity_pairs(rhyme_candidates, rhyme_mode=RHYME_MODE)

for pair in similarity_pairs:
    print(pair)

# Building a similarity matrix
similarity_matrix = build_similarity_matrix(rhyme_candidates, similarity_pairs)

for w1 in similarity_matrix:
    print(w1, similarity_matrix[w1])

In [ ]:
# Milestone 19a — passive logging: append every scored pair to the versioned dataset.
# Idempotent: re-running an unchanged verse appends nothing (see python/score_logging.py).
n_logged = log_scored_pairs(
    similarity_pairs,
    rhyme_candidates,
    DATASET_PATH,
    verse_id=VERSE_ID,
    rhyme_mode=RHYME_MODE,
    detection_mode=DETECTION_MODE,
)
print(f"Logged {n_logged} new scored pairs to {DATASET_PATH}")

In [ ]:
# Milestone 11 — panphon similarity spot-check
# Rhyme units below are real IPA sequences from english_mfa alignment of the
# project verse (see Section 4 output). They exercise the new feature-based
# scorer on perfect / slant / non-rhyme pairs.
from python.similarity_engine import rhyme_unit_similarity

spot_check_pairs = [
    # Perfect rhymes — expect 1.0
    ("found",  ["aw", "n", "d"],  "sound",  ["aw", "n", "d"]),
    ("light",  ["aj", "t"],       "night",  ["aj", "t"]),
    ("white",  ["aj", "ʈ"],       "tonight", ["aj", "ʈ"]),

    # Insertion slant (-oud vs -ound: one extra 'n') — expect a clear slant, not 0
    ("crowd",  ["aw", "d"],       "mound",  ["aw", "n", "d"]),

    # Substitution slant (t vs retroflex ʈ) — expect high
    ("light",  ["aj", "t"],       "white",  ["aj", "ʈ"]),

    # Genuinely different — expect low
    ("found",  ["aw", "n", "d"],  "light",  ["aj", "t"]),
    ("the",    ["a"],             "mound",  ["aw", "n", "d"]),
]

print(f"{'Word A':<10} {'Word B':<10} {'Score':>6}  Rhyme units")
print("-" * 55)
for wordA, ruA, wordB, ruB in spot_check_pairs:
    score = rhyme_unit_similarity(ruA, ruB)
    print(f"{wordA:<10} {wordB:<10} {score:>6.3f}  {ruA} | {ruB}")


---
## Section 6: Words Clustering <a id="clustering"></a>
---

In [ ]:
# Clustering by thresholds from the similarity matrix
# get_threshold() returns the correct value for the active RHYME_MODE
cluster_labels, clusters = cluster_rhymes(similarity_matrix, threshold=get_threshold(RHYME_MODE))

print("Clusters:")
print(clusters)

print("\nWord → Cluster Mapping:")
print(cluster_labels)

# Defining expected_clusters and comparing with the currect clustering
expected_clusters = {
    "underground": "A",
    "mound": "A",
    "flight": "B",
    "tonight": "B"
}

for r in rhyme_candidates:
    word = r["end_word"]
    r["expected_cluster"] = expected_clusters.get(word, None)

for r in rhyme_candidates:
    print(r)

# Exact rhymes clusters printout
clusters = cluster_labels

print(clusters)

# Debugging the clustering
debug_rhyme_analysis(rhyme_candidates, clusters)

# Accuracy evaluation
accuracy, mismatches = evaluate_rhyme_detection(rhyme_candidates, clusters)

In [ ]:
print(clusters)

In [ ]:
# Milestone 9 — full pipeline regression check
from python.similarity_engine import compute_similarity_pairs, build_similarity_matrix
from python.clustering import cluster_rhymes, get_threshold
from python.html_generation import filter_clusters

similarity_pairs = compute_similarity_pairs(rhyme_candidates, rhyme_mode=RHYME_MODE)
similarity_matrix = build_similarity_matrix(rhyme_candidates, similarity_pairs)
word_to_cluster, clusters = cluster_rhymes(similarity_matrix, threshold=get_threshold(RHYME_MODE))

passing_labels = filter_clusters(word_to_cluster, rhyme_candidates)

passing = {}
for entry in rhyme_candidates:
    word = entry["end_word"]
    label = word_to_cluster.get(word)
    if label in passing_labels:
        passing.setdefault(label, [])
        if word not in passing[label]:
            passing[label].append(word)

print("Passing clusters:")
for label, members in sorted(passing.items()):
    print(f"  {label}: {members}")

---
## Section 7: HTML Generation <a id="html-generation"></a>
---

In [ ]:
clusters = clusters = cluster_labels

generate_rhyme_html(rhyme_candidates, clusters, detection_mode=DETECTION_MODE, debug=True)

---
## Section 8: Syllable Engine (boundary-free) — primary view <a id="syllable-engine"></a>
---

Milestone 13. The **syllable** — not the word — is the atomic unit. Syllable
units cluster **freely across word and line boundaries**, so a long word can
participate in a scheme through any of its syllables, and a multi-syllable word
can wear several colours (one per syllable's cluster). This runs **in parallel**
to the word-based path in Sections 4–7, which is left untouched (it still feeds
the Milestone 19a pair-logging dataset).

In [ ]:
# Split every word into syllable units (one comparison unit per syllable), each
# carrying a back-pointer to its source word + phoneme range for colouring.
syllable_units = extract_syllable_candidates(
    INPUT_TXT_PATH,
    word_to_phonemes,
    detection_mode=DETECTION_MODE,
)

for u in syllable_units:
    print(u["unit_id"], u["source_word"], "".join(u["phonemes"]))

In [ ]:
# Score every pair of syllable units (role-aware panphon scorer: onset weight 0,
# nucleus full, coda discounted) and build the unit_id-keyed similarity matrix.
syllable_pairs = compute_syllable_pairs(syllable_units)
syllable_matrix = build_syllable_matrix(syllable_units, syllable_pairs)

print(f"{len(syllable_units)} syllable units, {len(syllable_pairs)} pairs scored")

In [ ]:
# Cluster the syllable units (average-linkage, effective-length scorer,
# threshold 0.65). Clustering is keyed on each syllable's unit_id, so syllables
# group freely across word and line boundaries.
syllable_labels, syllable_clusters = cluster_rhymes(
    syllable_matrix, threshold=SYLLABLE_THRESHOLD
)

for label in sorted(syllable_clusters):
    members = [
        f'{u["source_word"]}·{u["syllable_index"]}'
        for u in syllable_units
        if syllable_labels[u["unit_id"]] == label
    ]
    print(f"{label}: {members}")

In [ ]:
# Per-syllable colour rendering: each syllable wears its cluster's colour, so a
# multi-syllable word can carry several colours (the "Verse DNA" texture).
generate_syllable_html(syllable_units, syllable_labels, debug=True)